# Aggregate MCS — every repo run at a fixed 2-year window, no recency weight

One Model Confidence Set (Hansen, Lunde & Nason 2011) at the **5% level** over **every
distinct completed model** in the repo, each collapsed to a single forecast at a fixed
**2-year rolling window (504 days)** with **no recency weighting (δ=1.0)**. Log runs are
back-transformed to levels with Duan smearing (at δ=1.0 the weighted Duan equals the
unweighted `mean(exp(resid))`). QLIKE is in levels vs the same actual `RV_{t+1}` for
every model, so the losses are comparable.

**35 models**: 14 level-HAR OLS (±macro) + 14 log-HAR OLS (±macro) + 3 log-HAR OLS
cross-asset IV / all-exog + 4 elastic-net (walk-forward-CV). Saved to
`all_runs_mcs_2y.csv`.

In [1]:
# ===========================================================================
# Cell 1 — Imports & data
# ===========================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet, enet_path
from sklearn.preprocessing import StandardScaler

data = pd.read_parquet("merged_RV_GVZ_with_macro_event.parquet")  # RV_gold, RV_crude, RV_ES, GVZ_close, macro_event
iv   = pd.read_parquet("cross_asset_iv.parquet")                 # VIX_close, OVX_close
data = data.join(iv, how="inner").sort_index()                   # common Date index
rv = data["RV_gold"].astype(float)

TRADING_DAYS = 252
WINDOW = 504            # 2 years, fixed for every model
DELTA  = 1.0            # no recency weighting (unweighted OLS / EN)
EPS    = 1e-6           # QLIKE forecast floor

# Elastic-net selection grid (mirrors HAR_RegressionElasticnet3D.ipynb)
L1_RATIO = 0.1
ALPHAS   = np.logspace(-4, 1, 25)
WF_EDGES = [0.40, 0.60, 0.80, 1.00]

print(f"merged panel: {len(data)} obs, {data.index.min().date()} .. {data.index.max().date()}")
print(f"columns: {list(data.columns)}")
print(f"WINDOW={WINDOW} (2y), DELTA={DELTA} (no recency weight)")

merged panel: 4015 obs, 2010-06-08 .. 2026-05-29
columns: ['RV_gold', 'RV_crude', 'RV_ES', 'GVZ_close', 'macro_event', 'VIX_close', 'OVX_close']
WINDOW=504 (2y), DELTA=1.0 (no recency weight)


In [2]:
# ===========================================================================
# Cell 2 — Master design tables (level + log), one shared index
# ===========================================================================
for col in ["RV_gold", "GVZ_close", "RV_ES", "RV_crude", "VIX_close", "OVX_close"]:
    assert (data[col] > 0).all(), f"{col} has non-positive values; log undefined"

x     = np.log(rv)
macro = data["macro_event"].shift(-1).astype(float)   # day-(t+1) scheduled-release dummy

def _finish(df):
    df["y_log"]   = x.shift(-1)        # log(RV_{t+1})
    df["y_level"] = rv.shift(-1)       # RV_{t+1} in levels (QLIKE target for all models)
    return df.dropna()

# LOG master: log-HAR core + all logged exog + macro
LOG = pd.DataFrame(index=rv.index)
LOG["x_d"] = x
LOG["x_w"] = x.rolling(5).mean()
LOG["x_m"] = x.rolling(22).mean()
LOG["log_GVZ"]      = np.log(data["GVZ_close"])
LOG["log_RV_ES"]    = np.log(data["RV_ES"])
LOG["log_RV_crude"] = np.log(data["RV_crude"])
LOG["log_VIX"]      = np.log(data["VIX_close"])
LOG["log_OVX"]      = np.log(data["OVX_close"])
LOG["macro"]        = macro
LOG = _finish(LOG)

# LEVEL master: level-HAR core + all level exog + macro
LEV = pd.DataFrame(index=rv.index)
LEV["RV_d"] = rv
LEV["RV_w"] = rv.rolling(5).mean()
LEV["RV_m"] = rv.rolling(22).mean()
LEV["GVZ_close"] = data["GVZ_close"]
LEV["RV_ES"]     = data["RV_ES"]
LEV["RV_crude"]  = data["RV_crude"]
LEV["VIX_close"] = data["VIX_close"]
LEV["OVX_close"] = data["OVX_close"]
LEV["macro"]     = macro
LEV = _finish(LEV)

assert LOG.index.equals(LEV.index), "level/log master indices misaligned"
assert LOG.notna().all().all() and LEV.notna().all().all()
assert set(np.unique(LOG["macro"].to_numpy())) <= {0.0, 1.0}
print(f"LOG master: {LOG.shape}  cols={list(LOG.columns)}")
print(f"LEV master: {LEV.shape}  cols={list(LEV.columns)}")
LOG.head()

LOG master: (3993, 11)  cols=['x_d', 'x_w', 'x_m', 'log_GVZ', 'log_RV_ES', 'log_RV_crude', 'log_VIX', 'log_OVX', 'macro', 'y_log', 'y_level']
LEV master: (3993, 11)  cols=['RV_d', 'RV_w', 'RV_m', 'GVZ_close', 'RV_ES', 'RV_crude', 'VIX_close', 'OVX_close', 'macro', 'y_log', 'y_level']


,x_d,x_w,x_m,log_GVZ,log_RV_ES,log_RV_crude,log_VIX,log_OVX,macro,y_log,y_level
Date,,,,,,,,,,,
2010-07-08,2.702348,2.887899,2.773932,3.008648,2.948605,3.307625,3.246880,3.577389,0.0,2.723668,15.236102
2010-07-09,2.723668,2.800490,2.762134,2.979603,2.685524,3.119290,3.218076,3.542986,0.0,2.691347,14.751526
2010-07-12,2.691347,2.766723,2.755241,2.918311,2.779032,3.331252,3.195812,3.502550,0.0,2.593328,13.374209
2010-07-13,2.593328,2.694457,2.744754,2.943913,2.925677,3.260566,3.201119,3.492560,0.0,2.754495,15.713097
2010-07-14,2.754495,2.693037,2.755626,2.903617,2.912753,3.341328,3.214466,3.501344,1.0,2.722249,15.214503


In [3]:
# ===========================================================================
# Cell 3 — Helpers: QLIKE + one per-day loss-series function per model family
# ===========================================================================
START_DATE = LOG.index[WINDOW]   # 2y warm-up, common to every model


def _recency_weights(n, delta):
    if delta >= 1.0:
        return np.ones(n)
    ages = np.arange(n)[::-1]
    w = delta ** ages
    return w * (n / w.sum())

def _qlike(actual, forecast, eps=EPS):
    f = np.maximum(forecast, eps)
    r = actual / f
    return r - np.log(r) - 1.0, int((forecast <= eps).sum())

def _wproj_resid(A, Z, w):
    """Weighted residual of Z after projecting onto columns of A (W=diag(w))."""
    AtW = A.T * w
    coef = np.linalg.solve(AtW @ A, AtW @ Z)
    return Z - A @ coef, coef


# --- LEVEL OLS: target RV_{t+1}, linear forecast, NO smearing -----------------------
def rolling_level_ols_loss_series(design, feat_cols, window, start_date=None, delta=1.0):
    if start_date is None:
        start_date = START_DATE
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    lvl = design["y_level"].to_numpy()
    idx = design.index
    N, p = X.shape
    t_oos = np.arange(window, N)[idx[window:] >= start_date]
    starts = t_oos - window
    Xwins = np.lib.stride_tricks.sliding_window_view(X, window, axis=0)[starts].transpose(0, 2, 1)
    ywins = np.lib.stride_tricks.sliding_window_view(lvl, window)[starts]
    w  = _recency_weights(window, delta); sw = np.sqrt(w)
    Xs = Xwins * sw[None, :, None]; ys = ywins * sw[None, :]
    A = np.einsum("nwi,nwj->nij", Xs, Xs)
    b = np.einsum("nwi,nw->ni", Xs, ys)
    beta = np.linalg.solve(A, b)
    fc = np.einsum("np,np->n", X[t_oos], beta)        # linear forecast in levels
    q, _ = _qlike(lvl[t_oos], fc)
    return pd.Series(q, index=idx[t_oos], name="qlike")


# --- LOG OLS: target log RV_{t+1}, Duan smearing -> levels --------------------------
def rolling_log_ols_loss_series(design, feat_cols, window, start_date=None, delta=1.0):
    if start_date is None:
        start_date = START_DATE
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    N, p = X.shape
    t_oos = np.arange(window, N)[idx[window:] >= start_date]
    starts = t_oos - window
    Xwins = np.lib.stride_tricks.sliding_window_view(X, window, axis=0)[starts].transpose(0, 2, 1)
    ywins = np.lib.stride_tricks.sliding_window_view(yl, window)[starts]
    w  = _recency_weights(window, delta); sw = np.sqrt(w)
    Xs = Xwins * sw[None, :, None]; ys = ywins * sw[None, :]
    A = np.einsum("nwi,nwj->nij", Xs, Xs)
    b = np.einsum("nwi,nw->ni", Xs, ys)
    beta = np.linalg.solve(A, b)
    fitted = np.einsum("nwp,np->nw", Xwins, beta)
    smear = np.einsum("nw,w->n", np.exp(ywins - fitted), w) / w.sum()   # weighted Duan (=unweighted at delta=1)
    x_pred = np.einsum("np,np->n", X[t_oos], beta)
    fc = np.exp(x_pred) * smear
    q, _ = _qlike(lvl[t_oos], fc)
    return pd.Series(q, index=idx[t_oos], name="qlike")


# --- LOG elastic net (full penalty), per-day walk-forward-CV alpha -------------------
# Adapted from HAR_RegressionElasticnet3D.ipynb rolling_log_enet_wfcv_eval to return a
# per-day QLIKE Series. StandardScaler on all feats; enet_path CV by QLIKE-in-levels;
# refit ElasticNet at alpha* (recency-weighted); unweighted Duan smearing.
def rolling_log_enet_wfcv_loss_series(design, feat_cols, window, start_date=None,
                                      delta=1.0, l1_ratio=L1_RATIO, alphas=ALPHAS,
                                      wf_edges=WF_EDGES):
    if start_date is None:
        start_date = START_DATE
    Xf  = design[feat_cols].to_numpy()
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    w   = _recency_weights(window, delta)
    edges = [int(round(f * window)) for f in wf_edges]
    folds = [(0, edges[k], edges[k + 1]) for k in range(len(edges) - 1)]
    losses, dates, asel = [], [], []
    for t in range(window, len(design)):
        if idx[t] < start_date:
            continue
        Xw = Xf[t - window:t]; yw = yl[t - window:t]; lw = lvl[t - window:t]
        sc = StandardScaler().fit(Xw); Xw_s = sc.transform(Xw)
        qsum = np.zeros(len(alphas))
        for tr0, v0, v1 in folds:
            Xtr, ytr = Xw_s[tr0:v0], yw[tr0:v0]
            Xvl, lvl_vl = Xw_s[v0:v1], lw[v0:v1]
            w_tr = w[tr0:v0] * ((v0 - tr0) / w[tr0:v0].sum()); sw = np.sqrt(w_tr)
            xbar = np.average(Xtr, axis=0, weights=w_tr); ybar = np.average(ytr, weights=w_tr)
            a_out, coefs, _ = enet_path((Xtr - xbar) * sw[:, None], (ytr - ybar) * sw,
                                        l1_ratio=l1_ratio, alphas=alphas, max_iter=5000)
            tr_fit  = (Xtr - xbar) @ coefs + ybar
            smear   = np.mean(np.exp(ytr[:, None] - tr_fit), axis=0)
            val_log = (Xvl - xbar) @ coefs + ybar
            val_fc  = np.maximum(np.exp(val_log) * smear, EPS)
            r = lvl_vl[:, None] / val_fc
            qsum += np.mean(r - np.log(r) - 1.0, axis=0)
        a_star = float(a_out[int(np.argmin(qsum))])
        m = ElasticNet(alpha=a_star, l1_ratio=l1_ratio, fit_intercept=True,
                       max_iter=5000).fit(Xw_s, yw, sample_weight=w)
        x_hat = m.predict(sc.transform(Xf[t:t + 1]))[0]
        smearing = np.mean(np.exp(yw - m.predict(Xw_s)))
        fc = np.exp(x_hat) * smearing
        q, _ = _qlike(np.array([lvl[t]]), np.array([fc]))
        losses.append(q[0]); dates.append(idx[t]); asel.append(a_star)
    return pd.Series(losses, index=pd.DatetimeIndex(dates), name="qlike"), np.array(asel)


# --- LOG partial elastic net: penalise EXOG block only, HAR core unpenalised (FWL) --
def rolling_log_partial_enet_wfcv_loss_series(design, har_cols, exog_cols, window,
                                              start_date=None, delta=1.0,
                                              l1_ratio=L1_RATIO, alphas=ALPHAS,
                                              wf_edges=WF_EDGES):
    if start_date is None:
        start_date = START_DATE
    Ah  = np.column_stack([np.ones(len(design)), design[har_cols].to_numpy()])
    Bf  = design[exog_cols].to_numpy()
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    w   = _recency_weights(window, delta)
    edges = [int(round(f * window)) for f in wf_edges]
    folds = [(0, edges[k], edges[k + 1]) for k in range(len(edges) - 1)]
    losses, dates, asel = [], [], []
    for t in range(window, len(design)):
        if idx[t] < start_date:
            continue
        Aw = Ah[t - window:t]; yw = yl[t - window:t]; lw = lvl[t - window:t]
        sc = StandardScaler().fit(Bf[t - window:t]); Bw = sc.transform(Bf[t - window:t])
        qsum = np.zeros(len(alphas))
        for tr0, v0, v1 in folds:
            A_tr, B_tr, y_tr = Aw[tr0:v0], Bw[tr0:v0], yw[tr0:v0]
            A_vl, B_vl, l_vl = Aw[v0:v1], Bw[v0:v1], lw[v0:v1]
            w_tr = w[tr0:v0] * ((v0 - tr0) / w[tr0:v0].sum()); sw_tr = np.sqrt(w_tr)
            ry, _ = _wproj_resid(A_tr, y_tr, w_tr)
            RB, _ = _wproj_resid(A_tr, B_tr, w_tr)
            _, coefs, _ = enet_path(RB * sw_tr[:, None], ry * sw_tr,
                                    l1_ratio=l1_ratio, alphas=alphas, max_iter=5000)
            AtW = A_tr.T * w_tr
            b1 = np.linalg.solve(AtW @ A_tr, AtW @ (y_tr[:, None] - B_tr @ coefs))
            tr_fit  = A_tr @ b1 + B_tr @ coefs
            smear   = np.mean(np.exp(y_tr[:, None] - tr_fit), axis=0)
            val_log = A_vl @ b1 + B_vl @ coefs
            val_fc  = np.maximum(np.exp(val_log) * smear, EPS)
            r = l_vl[:, None] / val_fc
            qsum += np.mean(r - np.log(r) - 1.0, axis=0)
        a_star = float(alphas[int(np.argmin(qsum))])
        ry_f, _ = _wproj_resid(Aw, yw, w)
        RB_f, _ = _wproj_resid(Aw, Bw, w)
        m = ElasticNet(alpha=a_star, l1_ratio=l1_ratio, fit_intercept=False,
                       max_iter=5000).fit(RB_f, ry_f, sample_weight=w)
        b2 = m.coef_
        AtW = Aw.T * w
        b1 = np.linalg.solve(AtW @ Aw, AtW @ (yw - Bw @ b2))
        smear = np.mean(np.exp(yw - (Aw @ b1 + Bw @ b2)))
        x_b = sc.transform(Bf[t:t + 1])[0]
        fc = np.exp(Ah[t] @ b1 + x_b @ b2) * smear
        q, _ = _qlike(np.array([lvl[t]]), np.array([fc]))
        losses.append(q[0]); dates.append(idx[t]); asel.append(a_star)
    return pd.Series(losses, index=pd.DatetimeIndex(dates), name="qlike"), np.array(asel)


print(f"Common OOS start: {START_DATE.date()}  "
      f"({int((LOG.index >= START_DATE).sum())} forecast days)")

Common OOS start: 2012-07-06  (3489 forecast days)


In [4]:
# ===========================================================================
# Cell 4 — Registry of 35 models, single MCS, save all_runs_mcs_2y.csv
# ===========================================================================
from arch.bootstrap import MCS

LH = ["RV_d", "RV_w", "RV_m"]   # level-HAR core
XH = ["x_d", "x_w", "x_m"]      # log-HAR core
# level / log exogenous feature-combo suffixes (label, level-cols, log-cols)
COMBOS = [
    ("",                    [],                          []),
    (" + GVZ",              ["GVZ_close"],               ["log_GVZ"]),
    (" + SPX RV",          ["RV_ES"],                   ["log_RV_ES"]),
    (" + GVZ + SPX RV",    ["GVZ_close", "RV_ES"],      ["log_GVZ", "log_RV_ES"]),
    (" + Crude RV",        ["RV_crude"],                ["log_RV_crude"]),
    (" + GVZ + Crude RV",  ["GVZ_close", "RV_crude"],   ["log_GVZ", "log_RV_crude"]),
    (" + GVZ + SPX RV + Crude RV", ["GVZ_close", "RV_ES", "RV_crude"], ["log_GVZ", "log_RV_ES", "log_RV_crude"]),
]

registry = []   # (label, family, design, feat_cols [, exog_cols])
for suffix, lev_x, log_x in COMBOS:
    for mtag, mcol in [("", []), (" + macro", ["macro"])]:
        registry.append((f"Level HAR{suffix}{mtag}", "level-OLS", LEV, LH + lev_x + mcol))
        registry.append((f"Log HAR{suffix}{mtag}",   "log-OLS",   LOG, XH + log_x + mcol))

# cross-asset IV / all-exog (log OLS, as completed)
ALL_EXOG = ["log_GVZ", "log_VIX", "log_OVX", "log_RV_ES", "log_RV_crude", "macro"]
registry += [
    ("Log HAR + VIX",              "log-OLS", LOG, XH + ["log_VIX"]),
    ("Log HAR + OVX",              "log-OLS", LOG, XH + ["log_OVX"]),
    ("Log HAR + all-exog (OLS)",   "log-OLS", LOG, XH + ALL_EXOG),
]

# elastic-net variants (walk-forward-CV, delta=1.0)
EN_MODELS = [
    ("Log HAR + GVZ (EN)",             XH + ["log_GVZ"]),
    ("Log HAR + GVZ + SPX RV (EN)",    XH + ["log_GVZ", "log_RV_ES"]),
    ("Log HAR + GVZ + Crude RV (EN)",  XH + ["log_GVZ", "log_RV_crude"]),
]

print(f"Building {len(registry) + len(EN_MODELS) + 1} models...")
loss_cols, fam = {}, {}
for label, family, design, feats in registry:
    if family == "level-OLS":
        loss_cols[label] = rolling_level_ols_loss_series(design, feats, WINDOW, START_DATE, DELTA)
    else:
        loss_cols[label] = rolling_log_ols_loss_series(design, feats, WINDOW, START_DATE, DELTA)
    fam[label] = family

alpha_diag = {}
for label, feats in EN_MODELS:
    s, asel = rolling_log_enet_wfcv_loss_series(LOG, feats, WINDOW, START_DATE, DELTA)
    loss_cols[label] = s; fam[label] = "EN-full"; alpha_diag[label] = np.median(asel)

s, asel = rolling_log_partial_enet_wfcv_loss_series(LOG, XH, ALL_EXOG, WINDOW, START_DATE, DELTA)
loss_cols["Log HAR + all-exog (partial EN)"] = s
fam["Log HAR + all-exog (partial EN)"] = "EN-partial"; alpha_diag["Log HAR + all-exog (partial EN)"] = np.median(asel)

losses = pd.DataFrame(loss_cols).dropna()
assert losses.notna().all().all() and len(losses) > 0
print(f"\nMCS loss matrix: {losses.shape[0]} OOS days x {losses.shape[1]} models "
      f"({losses.index.min().date()} .. {losses.index.max().date()})")
print("EN median selected alphas:", {k: round(v, 4) for k, v in alpha_diag.items()})

mcs = MCS(losses, size=0.05, reps=10000, block_size=None,
          method="max", bootstrap="stationary", seed=42)
mcs.compute()

pv = mcs.pvalues["Pvalue"]
mcs_results = (pd.DataFrame({
        "model": list(losses.columns),
        "family": [fam[c] for c in losses.columns],
        "mean_qlike": [losses[c].mean() for c in losses.columns],
        "pvalue": [float(pv[c]) for c in losses.columns],
        "in_mcs": [bool(pv[c] > 0.05) for c in losses.columns],
    })
    .sort_values("mean_qlike", ascending=True).reset_index(drop=True))
mcs_results.to_csv("all_runs_mcs_2y.csv", index=False)

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
pd.set_option("display.max_rows", 60)
print(f"\n{int(mcs_results['in_mcs'].sum())} of {len(mcs_results)} models in the 5% MCS; "
      f"{int((~mcs_results['in_mcs']).sum())} excluded (p<=0.05).")
print(f"Best (lowest mean QLIKE): {mcs_results.iloc[0]['model']}  ({mcs_results.iloc[0]['mean_qlike']:.6f})")
print("Saved -> all_runs_mcs_2y.csv\n")
mcs_results

Building 35 models...



MCS loss matrix: 3489 OOS days x 35 models (2012-07-06 .. 2026-05-28)
EN median selected alphas: {'Log HAR + GVZ (EN)': 0.0316, 'Log HAR + GVZ + SPX RV (EN)': 0.0316, 'Log HAR + GVZ + Crude RV (EN)': 0.0511, 'Log HAR + all-exog (partial EN)': 0.0316}



13 of 35 models in the 5% MCS; 22 excluded (p<=0.05).
Best (lowest mean QLIKE): Log HAR + GVZ + macro  (0.029293)
Saved -> all_runs_mcs_2y.csv



,model,family,mean_qlike,pvalue,in_mcs
0,Log HAR + GVZ + macro,log-OLS,0.029293,1.000000,True
1,Log HAR + GVZ + Crude RV + macro,log-OLS,0.029337,0.616700,True
2,Log HAR + GVZ + SPX RV + macro,log-OLS,0.029433,0.198600,True
3,Log HAR + GVZ + SPX RV + Crude RV + macro,log-OLS,0.029486,0.147500,True
4,Log HAR + all-exog (OLS),log-OLS,0.029897,0.147500,True
5,Level HAR + GVZ + macro,level-OLS,0.030012,0.147500,True
6,Level HAR + GVZ + SPX RV + macro,level-OLS,0.030199,0.147500,True
7,Level HAR + GVZ + Crude RV + macro,level-OLS,0.030217,0.147500,True
8,Level HAR + GVZ + SPX RV + Crude RV + macro,level-OLS,0.030267,0.147500,True
9,Log HAR + GVZ,log-OLS,0.030468,0.147500,True
